# Session 2: RAG + Document Intelligence

**AI Agents Workshop Series — MQE Quant Lab, Spring 2026**

---

### What You'll Build Today

By the end of this session, you will have:

1. A **vector store** containing chunked and embedded FOMC meeting minutes
2. A **retrieval tool** that finds relevant passages using semantic search
3. A **RAG agent** that combines live FRED data with document search to answer policy questions grounded in primary sources
4. A simple **evaluation harness** to measure retrieval and answer quality

### Why RAG Matters for Quant Economics

Language models have a knowledge cutoff — they can't read last week's FOMC minutes, a new BLS methodology note, or an unpublished working paper. RAG (Retrieval-Augmented Generation) solves this by giving the model access to a searchable document store. The model retrieves relevant passages, then reasons over them alongside any other tools you've provided.

This is the same architecture behind Bloomberg's research assistants, sell-side "AI analyst" products, and internal knowledge systems at central banks.

### Prerequisites

- Completed Session 1 (working FRED agent)
- Python 3.11+, Anthropic API key, FRED API key

### Notebook Structure

| Section | Topic | Time |
|---------|-------|------|
| §1 | How RAG works — the five-stage pipeline | 10 min |
| §2 | Document loading and chunking strategies | 15 min |
| §3 | Embeddings and the vector store | 15 min |
| §4 | Building the RAG agent | 20 min |
| §5 | Evaluation and exercises | 15 min |

---

## §0 — Environment Setup

In [ ]:
# Install required packages (uncomment if needed)
# !pip install anthropic fredapi chromadb pandas python-dotenv requests

In [ ]:
import os
import json
import hashlib
import textwrap
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

assert os.environ.get("ANTHROPIC_API_KEY"), "Missing ANTHROPIC_API_KEY"
assert os.environ.get("FRED_API_KEY"), "Missing FRED_API_KEY"

import anthropic
import fredapi
import chromadb
import pandas as pd
import requests

client = anthropic.Anthropic()
fred = fredapi.Fred(api_key=os.environ["FRED_API_KEY"])

print(f"✓ anthropic {anthropic.__version__}")
print(f"✓ chromadb  {chromadb.__version__}")
print("✓ All API keys loaded.")

---

## §1 — How RAG Works: The Five-Stage Pipeline

RAG separates into two phases: an **indexing** phase (done once, ahead of time) and a **query** phase (done at inference for each question).

### Indexing Phase (Offline)

```
Raw Documents
  → [1] LOAD:  Read PDFs, HTML, plaintext
  → [2] CHUNK: Split into passages (500-1000 tokens each)
  → [3] EMBED: Convert each chunk to a vector (dense numerical representation)
  → [4] STORE: Insert vectors + text into a vector database
```

### Query Phase (Online)

```
User Question
  → [5] RETRIEVE: Embed the question → find nearest chunks in the vector store
  → [6] GENERATE: Pass retrieved chunks + question to the LLM → grounded answer
```

### Key design decisions

Each stage has parameters that affect quality:

| Stage | Decision | Impact |
|-------|----------|--------|
| Chunk | Size (words/tokens) | Too small → fragments lose context. Too large → dilutes relevance. |
| Chunk | Overlap | Prevents important sentences from being split across chunks. |
| Embed | Model choice | Determines the quality of semantic similarity matching. |
| Retrieve | Number of results (k) | Too few → misses relevant context. Too many → floods the LLM with noise. |

We'll make each of these decisions explicitly, so you understand the tradeoffs.

---

## §2 — Document Loading and Chunking

We'll work with FOMC meeting minutes — the primary source document that markets parse for monetary policy signals. These are public, text-heavy, and directly relevant to macro research.

### 2.1 — Download FOMC Minutes

The Federal Reserve publishes meeting minutes as HTML on its website. We'll download three recent sets of minutes and extract the text.

In [ ]:
# FOMC minutes URLs — we'll use 3 recent meetings
# These are real URLs from the Federal Reserve Board website.
# If a URL stops working, replace it with any recent minutes page from:
# https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm

FOMC_SOURCES = [
    {
        "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20250129.htm",
        "meeting": "January 2025",
        "label": "jan_2025",
    },
    {
        "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20250319.htm",
        "meeting": "March 2025",
        "label": "mar_2025",
    },
    {
        "url": "https://www.federalreserve.gov/monetarypolicy/fomcminutes20250507.htm",
        "meeting": "May 2025",
        "label": "may_2025",
    },
]

In [ ]:
import re
from html import unescape


def fetch_fomc_minutes(url: str) -> str:
    """
    Download FOMC minutes from the Fed website and extract clean text.

    The Fed publishes minutes as HTML. We strip tags, collapse whitespace,
    and return the body text. This is a simple extraction — for production
    use, you'd use a proper HTML-to-text library like trafilatura.

    Parameters
    ----------
    url : str
        Full URL to the FOMC minutes page.

    Returns
    -------
    str
        Cleaned plaintext of the minutes.
    """
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    html = response.text

    # Extract the main content div (Fed pages use a consistent structure)
    # Try to isolate the article body
    match = re.search(
        r'<div[^>]*id="article"[^>]*>(.*?)</div>\s*<!--\s*end\s+article',
        html,
        re.DOTALL | re.IGNORECASE,
    )
    if match:
        html = match.group(1)

    # Remove script and style blocks
    html = re.sub(r"<(script|style)[^>]*>.*?</\1>", "", html, flags=re.DOTALL | re.IGNORECASE)

    # Replace paragraph and break tags with newlines
    html = re.sub(r"<(?:p|br|div|h[1-6])[^>]*>", "\n", html, flags=re.IGNORECASE)

    # Strip remaining HTML tags
    text = re.sub(r"<[^>]+>", "", html)

    # Decode HTML entities and normalize whitespace
    text = unescape(text)
    text = re.sub(r"[ \t]+", " ", text)         # collapse horizontal whitespace
    text = re.sub(r"\n{3,}", "\n\n", text)       # collapse excessive newlines
    text = text.strip()

    return text

In [ ]:
# Download all three sets of minutes
documents = {}

for source in FOMC_SOURCES:
    try:
        text = fetch_fomc_minutes(source["url"])
        documents[source["label"]] = {
            "text": text,
            "meeting": source["meeting"],
            "url": source["url"],
        }
        print(f"✓ {source['meeting']:15s}  {len(text):,} chars  ({len(text.split()):,} words)")
    except Exception as e:
        print(f"✗ {source['meeting']:15s}  Failed: {e}")

print(f"\nLoaded {len(documents)} document(s).")

> **If the downloads fail** (the Fed may change URLs), you can use any plaintext documents instead. The RAG pipeline works the same regardless of content. Place `.txt` files in a `data/` directory and load them with `Path('data/').read_text()`.

Let's inspect what we downloaded:

In [ ]:
# Preview the first document
sample_key = list(documents.keys())[0]
sample_text = documents[sample_key]["text"]

print(f"Document: {documents[sample_key]['meeting']}")
print(f"Length:   {len(sample_text):,} characters")
print(f"\n--- First 1000 characters ---\n")
print(sample_text[:1000])

### 2.2 — Chunking Strategies

We need to split each document into passages small enough for the embedding model (which has a token limit) and focused enough to be useful when retrieved. There are several strategies:

| Strategy | How it works | Best for |
|----------|-------------|----------|
| Fixed word window | Split every N words with overlap | Simple, general-purpose |
| Paragraph-based | Split on double newlines | Documents with natural structure |
| Recursive character | Split by hierarchy: paragraphs → sentences → words | LangChain's default |
| Semantic chunking | Use embeddings to find topic boundaries | Research papers, long reports |

FOMC minutes have a clear paragraph structure, so we'll use **paragraph-based chunking with a size constraint**: we split on paragraph boundaries, then merge small paragraphs together until we hit a target size.

This preserves the document's natural structure while keeping chunks in the right size range.

In [ ]:
def chunk_by_paragraphs(
    text: str,
    target_words: int = 300,
    max_words: int = 500,
    overlap_words: int = 50,
) -> list[str]:
    """
    Chunk text by paragraph boundaries with size constraints.

    Algorithm:
    1. Split text into paragraphs (on double newlines)
    2. Accumulate paragraphs into a chunk until target_words is reached
    3. If a single paragraph exceeds max_words, split it by sentences
    4. Add overlap from the end of the previous chunk

    Parameters
    ----------
    text : str
        Full document text.
    target_words : int
        Ideal chunk size in words. Chunks may be slightly larger
        to avoid splitting mid-paragraph.
    max_words : int
        Hard upper limit. Paragraphs exceeding this are split.
    overlap_words : int
        Number of words from the end of the previous chunk to prepend.

    Returns
    -------
    list[str]
        List of text chunks.
    """
    # Split into paragraphs
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]

    chunks = []
    current_chunk_parts = []
    current_word_count = 0

    for para in paragraphs:
        para_words = len(para.split())

        # If a single paragraph is very long, split it by sentences
        if para_words > max_words:
            # Flush the current chunk first
            if current_chunk_parts:
                chunks.append("\n\n".join(current_chunk_parts))
                current_chunk_parts = []
                current_word_count = 0

            sentences = re.split(r"(?<=[.!?])\s+", para)
            sent_chunk = []
            sent_word_count = 0
            for sent in sentences:
                sent_words = len(sent.split())
                if sent_word_count + sent_words > target_words and sent_chunk:
                    chunks.append(" ".join(sent_chunk))
                    sent_chunk = []
                    sent_word_count = 0
                sent_chunk.append(sent)
                sent_word_count += sent_words
            if sent_chunk:
                chunks.append(" ".join(sent_chunk))
            continue

        # Would adding this paragraph exceed the target?
        if current_word_count + para_words > target_words and current_chunk_parts:
            chunks.append("\n\n".join(current_chunk_parts))
            current_chunk_parts = []
            current_word_count = 0

        current_chunk_parts.append(para)
        current_word_count += para_words

    # Don't forget the last chunk
    if current_chunk_parts:
        chunks.append("\n\n".join(current_chunk_parts))

    # Add overlap: prepend the last N words of the previous chunk
    if overlap_words > 0 and len(chunks) > 1:
        chunks_with_overlap = [chunks[0]]
        for i in range(1, len(chunks)):
            prev_words = chunks[i - 1].split()
            overlap_text = " ".join(prev_words[-overlap_words:])
            chunks_with_overlap.append(f"[...] {overlap_text}\n\n{chunks[i]}")
        return chunks_with_overlap

    return chunks

In [ ]:
# Chunk all documents
all_chunks = []  # list of dicts with text, metadata

for label, doc in documents.items():
    chunks = chunk_by_paragraphs(doc["text"], target_words=300, overlap_words=50)

    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "id": f"{label}_chunk_{i:03d}",
            "text": chunk,
            "metadata": {
                "meeting": doc["meeting"],
                "source": label,
                "chunk_index": i,
                "total_chunks": len(chunks),
                "url": doc["url"],
            },
        })

    print(f"{doc['meeting']:15s}  →  {len(chunks):3d} chunks")

print(f"\nTotal: {len(all_chunks)} chunks across {len(documents)} documents")

In [ ]:
# Inspect chunk statistics
word_counts = [len(c["text"].split()) for c in all_chunks]
s = pd.Series(word_counts)

print("Chunk size distribution (words):")
print(f"  Min:    {s.min()}")
print(f"  Median: {s.median():.0f}")
print(f"  Mean:   {s.mean():.0f}")
print(f"  Max:    {s.max()}")
print(f"  Std:    {s.std():.0f}")

print(f"\n--- Sample chunk ({all_chunks[3]['id']}) ---\n")
print(textwrap.fill(all_chunks[3]["text"][:600], width=90))
print(f"\n  Metadata: {all_chunks[3]['metadata']}")

> **Design note:** Our chunks average around 250–350 words. This is a good size for economic text: large enough to contain a complete argument (e.g., "participants noted that inflation remained elevated… several members expressed concern about…"), but small enough that retrieved results are focused and relevant.
>
> The overlap ensures that if an important sentence falls at a chunk boundary, it appears in both adjacent chunks — so retrieval won't miss it.

---

## §3 — Embeddings and the Vector Store

### 3.1 — What Are Embeddings?

An **embedding** is a dense vector (e.g., 1024 floating-point numbers) that represents the *meaning* of a text passage. Passages with similar meaning have vectors that are close together in the embedding space.

```
"The Fed raised rates by 25 basis points"  →  [0.12, -0.34, 0.56, ...]
"Interest rates were increased by a quarter point"  →  [0.11, -0.33, 0.55, ...]
"Housing starts fell 3% in March"  →  [-0.45, 0.22, -0.18, ...]
```

The first two vectors will be close (cosine similarity ≈ 0.95) because they express the same idea. The third is distant because it's about a different topic.

We'll use **Claude's built-in embedding support via ChromaDB's default model**, which uses a lightweight local embedding model. For production, you'd use Voyage AI or OpenAI's embedding models for higher quality.

### 3.2 — Setting Up ChromaDB

ChromaDB is a vector database that handles embedding, storage, and retrieval. It runs locally — no external service needed.

In [ ]:
# Initialize ChromaDB with a persistent directory
# Data will survive kernel restarts if you use a path.
# Use chromadb.Client() for ephemeral (in-memory) storage during experiments.

chroma_client = chromadb.Client()  # in-memory for this workshop

# Delete the collection if it already exists (for re-running this notebook)
try:
    chroma_client.delete_collection("fomc_minutes")
except Exception:
    pass

# Create a fresh collection
# ChromaDB uses its built-in embedding function by default (all-MiniLM-L6-v2)
collection = chroma_client.create_collection(
    name="fomc_minutes",
    metadata={"hnsw:space": "cosine"},  # use cosine similarity for retrieval
)

print(f"✓ Collection 'fomc_minutes' created.")
print(f"  Embedding model: default (all-MiniLM-L6-v2, 384 dimensions)")
print(f"  Distance metric: cosine")

### 3.3 — Indexing: Embed and Store All Chunks

We pass our chunks to ChromaDB, which handles embedding automatically. Each chunk gets a unique ID and metadata that we can filter on later.

In [ ]:
# Batch insert all chunks into the collection
# ChromaDB embeds them automatically using the default model.

BATCH_SIZE = 50  # ChromaDB recommends batching for large inserts

for i in range(0, len(all_chunks), BATCH_SIZE):
    batch = all_chunks[i : i + BATCH_SIZE]

    collection.add(
        ids=[c["id"] for c in batch],
        documents=[c["text"] for c in batch],
        metadatas=[c["metadata"] for c in batch],
    )

    print(f"  Indexed batch {i // BATCH_SIZE + 1}: chunks {i}–{i + len(batch) - 1}")

print(f"\n✓ {collection.count()} chunks indexed in the vector store.")

### 3.4 — Retrieval: Semantic Search

Now we can search the vector store with a natural language query. ChromaDB embeds the query, finds the nearest chunk vectors, and returns them ranked by similarity.

Let's test retrieval directly before wiring it into the agent.

In [ ]:
def search_fomc(query: str, n_results: int = 5, meeting_filter: str = None) -> dict:
    """
    Search the FOMC minutes vector store.

    Parameters
    ----------
    query : str
        Natural language search query.
    n_results : int
        Number of results to return.
    meeting_filter : str or None
        If set, restrict search to a specific meeting (e.g., 'January 2025').

    Returns
    -------
    dict
        Raw ChromaDB query results with documents, metadatas, and distances.
    """
    where_filter = None
    if meeting_filter:
        where_filter = {"meeting": meeting_filter}

    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        where=where_filter,
        include=["documents", "metadatas", "distances"],
    )

    return results

In [ ]:
# Test: search for inflation-related passages
results = search_fomc("inflation expectations and price stability", n_results=3)

for i in range(len(results["ids"][0])):
    doc_id = results["ids"][0][i]
    distance = results["distances"][0][i]
    meeting = results["metadatas"][0][i]["meeting"]
    text_preview = results["documents"][0][i][:300]

    print(f"\n{'─'*70}")
    print(f"Result {i+1}  |  {doc_id}  |  {meeting}  |  distance: {distance:.4f}")
    print(f"{'─'*70}")
    print(textwrap.fill(text_preview, width=90))
    print("...")

In [ ]:
# Test: search with a metadata filter — only one specific meeting
first_meeting = list(documents.values())[0]["meeting"]
results = search_fomc(
    "labor market conditions and employment",
    n_results=3,
    meeting_filter=first_meeting,
)

print(f"Searching only '{first_meeting}' minutes:\n")
for i in range(len(results["ids"][0])):
    print(f"  [{i+1}] {results['ids'][0][i]}  (dist: {results['distances'][0][i]:.4f})")
    print(f"      {results['documents'][0][i][:150]}...\n")

> **Observe:** The retrieval is *semantic*, not keyword-based. Searching for "labor market conditions" retrieves passages about employment, jobs, hiring — even if those exact words don't appear. This is the power of embedding-based search over traditional text search.

---

## §4 — Building the RAG Agent

Now we combine everything: the FRED data tool from Session 1, the FOMC search tool we just built, and the ReAct agent loop. The agent will decide which tool to use — or both — based on the question.

### 4.1 — Define Tools

We need to register both the FRED data tool (from Session 1) and the new FOMC search tool.

In [ ]:
# ── Tool 1: FRED data (carried forward from Session 1) ──

def get_fred_series(series_id: str, start_date: str = "2020-01-01", n_recent: int = 12) -> str:
    """Fetch an economic time series from FRED."""
    try:
        data = fred.get_series(series_id, observation_start=start_date)
        info = fred.get_series_info(series_id)
        recent = data.dropna().tail(n_recent)
        result = {
            "series_id": series_id,
            "title": info["title"],
            "units": info["units"],
            "frequency": info["frequency"],
            "observations": {
                str(date.date()): round(float(val), 4)
                for date, val in recent.items()
            },
        }
        return json.dumps(result, indent=2)
    except Exception as e:
        return json.dumps({"error": str(e), "series_id": series_id})


# ── Tool 2: FOMC minutes search (new) ──

def search_fomc_minutes(query: str, n_results: int = 5, meeting: str = None) -> str:
    """
    Search FOMC meeting minutes by semantic query.

    Returns relevant passages from recent FOMC minutes along with
    source metadata (which meeting, chunk ID, relevance score).
    """
    results = search_fomc(query, n_results=n_results, meeting_filter=meeting)

    passages = []
    for i in range(len(results["ids"][0])):
        passages.append({
            "meeting": results["metadatas"][0][i]["meeting"],
            "relevance_score": round(1.0 - results["distances"][0][i], 4),
            "text": results["documents"][0][i],
        })

    return json.dumps(passages, indent=2)

In [ ]:
# Tool schemas for Claude
RAG_TOOLS = [
    {
        "name": "get_fred_series",
        "description": (
            "Fetch an economic time series from the FRED database. "
            "Use for current/historical data on GDP, inflation, unemployment, "
            "interest rates, housing, employment, etc. "
            "Common series: GDP, UNRATE, CPIAUCSL, PCEPI, DFF, T10Y2Y, PAYEMS, HOUST."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "series_id": {
                    "type": "string",
                    "description": "FRED series ID (e.g., 'UNRATE')",
                },
                "start_date": {
                    "type": "string",
                    "description": "Start date YYYY-MM-DD (default: '2020-01-01')",
                },
                "n_recent": {
                    "type": "integer",
                    "description": "Number of recent observations (default: 12)",
                },
            },
            "required": ["series_id"],
        },
    },
    {
        "name": "search_fomc_minutes",
        "description": (
            "Search FOMC (Federal Open Market Committee) meeting minutes using semantic search. "
            "Returns relevant passages about monetary policy discussions, economic outlook, "
            "and committee decisions. Use this to find what the Fed said about specific topics "
            "like inflation, employment, rate decisions, or economic risks. "
            "Available meetings: " + ", ".join(d["meeting"] for d in documents.values()) + "."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Semantic search query about Fed policy or economic conditions",
                },
                "n_results": {
                    "type": "integer",
                    "description": "Number of passages to return (default: 5, max: 10)",
                },
                "meeting": {
                    "type": "string",
                    "description": "Optional: restrict to a specific meeting (e.g., 'January 2025')",
                },
            },
            "required": ["query"],
        },
    },
]

# Tool registry
TOOL_REGISTRY = {
    "get_fred_series": get_fred_series,
    "search_fomc_minutes": search_fomc_minutes,
}

### 4.2 — The RAG-Enhanced Agent Loop

This is the same `run_agent` pattern from Session 1, with an improved system prompt that instructs the agent to ground its answers in retrieved documents.

In [ ]:
RAG_SYSTEM_PROMPT = """You are a senior Federal Reserve policy analyst. Your job is to
answer questions about monetary policy and the economic outlook using two sources:

1. FRED DATA: Live economic time series (GDP, CPI, employment, rates, etc.)
2. FOMC MINUTES: Searchable text from recent Federal Open Market Committee meetings.

Rules:
- Always search FOMC minutes when a question involves Fed policy, rate decisions,
  or the committee's economic assessment. Do not rely on your training data for
  what the Fed said — retrieve it from the minutes.
- Always pull FRED data when a question involves specific economic indicators.
  Do not quote numbers from memory — fetch the actual data.
- When citing FOMC minutes, reference which meeting the passage comes from.
- Be precise. Include dates, numbers, and units.
- Distinguish between what the data shows and what the FOMC discussed.
- If the retrieved passages don't contain relevant information, say so.
"""


def run_rag_agent(
    user_query: str,
    model: str = "claude-sonnet-4-20250514",
    max_iterations: int = 10,
    verbose: bool = True,
) -> str:
    """
    Run the RAG agent with FRED + FOMC tools.

    This is the same ReAct loop from Session 1, now with two tools:
    live data (FRED) and document search (FOMC minutes).
    """
    messages = [{"role": "user", "content": user_query}]
    total_input_tokens = 0
    total_output_tokens = 0

    for iteration in range(max_iterations):
        response = client.messages.create(
            model=model,
            max_tokens=4096,
            system=RAG_SYSTEM_PROMPT,
            tools=RAG_TOOLS,
            messages=messages,
        )

        total_input_tokens += response.usage.input_tokens
        total_output_tokens += response.usage.output_tokens

        # Final answer
        if response.stop_reason == "end_turn":
            final_text = next(
                (b.text for b in response.content if b.type == "text"), ""
            )
            if verbose:
                cost = (
                    total_input_tokens * 3.0 / 1_000_000
                    + total_output_tokens * 15.0 / 1_000_000
                )
                print(f"\n{'='*60}")
                print(f"Agent complete in {iteration + 1} iteration(s)")
                print(f"Tokens: {total_input_tokens:,} in / {total_output_tokens:,} out")
                print(f"Estimated cost: ${cost:.4f}")
                print(f"{'='*60}\n")
            return final_text

        # Tool use
        if response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})

            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    if verbose:
                        args_preview = json.dumps(block.input)
                        if len(args_preview) > 100:
                            args_preview = args_preview[:100] + "..."
                        print(f"  [{iteration+1}] {block.name}({args_preview})")

                    func = TOOL_REGISTRY.get(block.name)
                    if func:
                        result = func(**block.input)
                    else:
                        result = json.dumps({"error": f"Unknown tool: {block.name}"})

                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    })

            messages.append({"role": "user", "content": tool_results})

    return "Agent reached maximum iterations without completing."

### 4.3 — Test the RAG Agent

Let's ask questions that require different combinations of tools.

**Question type 1: FOMC minutes only** — the agent should search the documents.

In [ ]:
answer = run_rag_agent(
    "What did the FOMC say about inflation in their most recent meeting? "
    "Were there any dissenting views?"
)
print(answer)

**Question type 2: FRED data only** — the agent should call the FRED tool.

In [ ]:
answer = run_rag_agent(
    "What has happened to the unemployment rate over the past 12 months?"
)
print(answer)

**Question type 3: Both tools** — the agent should retrieve FOMC context and pull live data.

In [ ]:
answer = run_rag_agent(
    "The FOMC has discussed the path of the federal funds rate in recent meetings. "
    "What was their language around rate trajectory, and how does it compare "
    "to what the actual Fed Funds rate (DFF) has done over the past 6 months?"
)
print(answer)

In [ ]:
# Harder question: cross-referencing FOMC language with multiple data series
answer = run_rag_agent(
    "Based on the FOMC minutes, what were the committee's main concerns "
    "about the labor market? Pull the actual data for nonfarm payrolls "
    "(PAYEMS) and the unemployment rate (UNRATE) to evaluate whether "
    "those concerns appear justified by the data."
)
print(answer)

> **Key insight:** The agent is now doing something that no single tool can do alone — it's combining *qualitative* information (what the Fed said) with *quantitative* information (what the data shows) and synthesizing a coherent analysis. This is exactly how a human analyst works, except the agent does it in seconds.
>
> Watch the tool call sequence in verbose mode: the agent typically searches the minutes first, then pulls relevant data to validate or contextualize what it found.

### 4.4 — Comparing Across Meetings

One strength of having multiple meetings indexed is the ability to track how the Fed's language has evolved. The agent can search multiple meetings and identify shifts in tone or emphasis.

In [ ]:
answer = run_rag_agent(
    "How has the FOMC's language about inflation evolved across the meetings "
    "available in the minutes? Search each meeting separately and identify "
    "any shifts in tone, concern, or policy direction."
)
print(answer)

---

## §5 — Evaluation and Exercises

### 5.1 — Evaluating Retrieval Quality

A RAG system is only as good as its retrieval. If the wrong chunks are retrieved, the LLM will generate a confident but wrong answer. We need to test whether retrieval is working.

The simplest evaluation: write test questions where you know which meeting (or which topic) the answer should come from, then check whether the retrieved chunks match.

This is called a **retrieval relevance test** — it checks the R in RAG before you even involve the generator.

In [ ]:
# Define test queries with expected metadata
# Adjust these based on the meetings you actually indexed.
# The key is: you know what meeting should be retrieved.

EVAL_QUERIES = [
    {
        "query": "federal funds rate target range decision",
        "expected_topic": "rate decision",
        "description": "Should retrieve passages about the rate decision itself",
    },
    {
        "query": "consumer price index inflation outlook",
        "expected_topic": "inflation",
        "description": "Should retrieve inflation discussion passages",
    },
    {
        "query": "labor market tightness hiring conditions",
        "expected_topic": "employment",
        "description": "Should retrieve labor market passages",
    },
    {
        "query": "risks to the economic outlook downside scenarios",
        "expected_topic": "risks",
        "description": "Should retrieve risk assessment passages",
    },
    {
        "query": "balance sheet reduction quantitative tightening",
        "expected_topic": "balance sheet",
        "description": "Should retrieve QT-related passages",
    },
]

In [ ]:
def evaluate_retrieval(queries: list[dict], n_results: int = 3) -> pd.DataFrame:
    """
    Run retrieval evaluation across a set of test queries.

    For each query, retrieves the top-k chunks and displays them
    for manual inspection. Returns a summary DataFrame.
    """
    rows = []

    for q in queries:
        results = search_fomc(q["query"], n_results=n_results)

        print(f"\n{'━'*70}")
        print(f"Query: {q['query']}")
        print(f"Expected topic: {q['expected_topic']}")
        print(f"{'━'*70}")

        for i in range(len(results["ids"][0])):
            distance = results["distances"][0][i]
            meeting = results["metadatas"][0][i]["meeting"]
            text = results["documents"][0][i][:200]

            relevance = 1.0 - distance  # convert distance to similarity

            print(f"  [{i+1}] similarity={relevance:.3f}  |  {meeting}")
            print(f"      {text}...")

            rows.append({
                "query": q["query"][:40],
                "expected_topic": q["expected_topic"],
                "rank": i + 1,
                "meeting": meeting,
                "similarity": round(relevance, 4),
            })

    return pd.DataFrame(rows)


eval_df = evaluate_retrieval(EVAL_QUERIES)
print("\n\n=== Summary ===")
print(f"Mean similarity (rank 1): {eval_df[eval_df['rank']==1]['similarity'].mean():.3f}")
print(f"Mean similarity (all):    {eval_df['similarity'].mean():.3f}")

> **How to read these results:**
>
> - **Similarity > 0.5** for the top result = good. The chunk is clearly relevant.
> - **Similarity < 0.3** = the query and the chunks aren't matching well. Consider re-phrasing the query or adjusting chunk size.
> - **Wrong meeting at rank 1** = the retrieval is pulling from the wrong document. Add more metadata filtering or refine chunk boundaries.
>
> This manual evaluation is the starting point. In production, you'd build automated relevance judgments using labeled examples.

### 5.2 — Evaluating End-to-End Answer Quality

Retrieval quality is necessary but not sufficient. We also need to check that the *generated answers* are accurate and grounded. A simple approach: ask the agent questions you can verify, then use Claude itself to grade the answers.

This is called **LLM-as-judge** evaluation — a practical technique when human labeling is expensive.

In [ ]:
def grade_answer(question: str, answer: str, criteria: str = None) -> dict:
    """
    Use Claude to evaluate the quality of a RAG agent's answer.

    Returns a structured evaluation with scores and reasoning.
    """
    grading_prompt = f"""You are evaluating the quality of a research assistant's answer.

Question asked: {question}

Answer given:
{answer}

Rate the answer on these dimensions (1-5 scale):
1. GROUNDED: Does it cite specific data or document sources, not vague claims? (1=no sources, 5=everything cited)
2. ACCURATE: Are the claims consistent and plausible? (1=contradictory/wrong, 5=precise and correct)
3. COMPLETE: Does it address all parts of the question? (1=ignores key parts, 5=fully addresses)
4. CONCISE: Is it focused without unnecessary filler? (1=rambling, 5=tight and efficient)

{f'Additional criteria: {criteria}' if criteria else ''}

Respond ONLY with valid JSON (no markdown, no backticks):
{{
  "grounded": <int 1-5>,
  "accurate": <int 1-5>,
  "complete": <int 1-5>,
  "concise": <int 1-5>,
  "overall": <float, average of the four scores>,
  "reasoning": "<one paragraph explaining the scores>"
}}"""

    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=512,
        messages=[{"role": "user", "content": grading_prompt}],
    )

    try:
        return json.loads(response.content[0].text)
    except json.JSONDecodeError:
        return {"error": "Failed to parse grading response", "raw": response.content[0].text}

In [ ]:
# Run the agent on a test question and grade the answer
test_question = (
    "What is the FOMC's current assessment of inflation, "
    "and what does the actual CPI data show?"
)

print("Running agent...\n")
test_answer = run_rag_agent(test_question, verbose=True)
print(test_answer)

print("\n\nGrading answer...\n")
grade = grade_answer(
    test_question,
    test_answer,
    criteria="The answer should cite specific FOMC meeting dates and actual CPI numbers.",
)

print(f"Grounded:  {grade.get('grounded', '?')}/5")
print(f"Accurate:  {grade.get('accurate', '?')}/5")
print(f"Complete:  {grade.get('complete', '?')}/5")
print(f"Concise:   {grade.get('concise', '?')}/5")
print(f"Overall:   {grade.get('overall', '?')}/5")
print(f"\nReasoning: {grade.get('reasoning', '?')}")

---

### Exercises

### Exercise 1: Tune Chunk Parameters (10 min)

Rebuild the vector store with different chunking parameters and compare retrieval quality.

**Steps:**
1. Create two new collections: `fomc_small_chunks` (target=150 words) and `fomc_large_chunks` (target=500 words)
2. Re-chunk and index the documents into each collection
3. Run the same 5 evaluation queries against both collections
4. Compare: which chunk size gives better top-1 similarity? Which retrieves more focused passages?

**What to look for:** Small chunks are more precise but may lose context. Large chunks retain more context but the retrieved text may include irrelevant paragraphs.

In [ ]:
# Exercise 1: Your code here
#
# small_collection = chroma_client.create_collection("fomc_small_chunks", ...)
# large_collection = chroma_client.create_collection("fomc_large_chunks", ...)
#
# For each: chunk_by_paragraphs(text, target_words=...) → index → evaluate
#
# Tip: modify search_fomc() to accept a collection parameter, then compare.

### Exercise 2: Add Metadata Filtering (10 min)

Modify the `search_fomc_minutes` tool so the agent can filter by meeting date. Then test with a question like:

*"What did the FOMC say about housing in their January 2025 meeting specifically?"*

The agent should pass the `meeting` parameter to restrict the search.

**Bonus:** Add a `list_available_meetings` tool that returns the set of meetings in the vector store. This lets the agent discover what's available before searching.

In [ ]:
# Exercise 2: Your code here
#
# The search_fomc_minutes function already supports a `meeting` parameter.
# The tool schema already declares it. Test whether the agent uses it
# when the question references a specific meeting.
#
# For the bonus: define a list_available_meetings tool that queries
# the collection metadata.

### Exercise 3: Multi-Document Comparison Agent (15 min)

Build a prompt (modify `RAG_SYSTEM_PROMPT`) that instructs the agent to:

1. Search each meeting separately for the same topic
2. Compare the language across meetings
3. Identify hawkish vs. dovish shifts

Test with: *"Track the evolution of the FOMC's inflation language across all available meetings. Has the committee become more or less concerned?"*

**Hint:** You may need to increase `n_results` or make multiple search calls.

In [ ]:
# Exercise 3: Your code here
#
# COMPARISON_SYSTEM_PROMPT = """..."""
#
# answer = run_rag_agent(
#     "Track the evolution of...",
#     system=COMPARISON_SYSTEM_PROMPT,
# )

### Exercise 4 (Challenge): Automated Evaluation Pipeline (15 min)

Build a function `run_eval_suite()` that:

1. Takes a list of test questions
2. Runs the RAG agent on each
3. Grades each answer with `grade_answer()`
4. Returns a DataFrame with scores across all questions
5. Computes aggregate metrics (mean score per dimension)

This is the pattern you'd use to compare different chunking strategies, embedding models, or system prompts.

In [ ]:
# Exercise 4: Your code here
#
# EVAL_QUESTIONS = [
#     "What is the FOMC's current stance on rate cuts?",
#     "What does the committee say about labor market risks?",
#     "How does actual GDP growth compare to the Fed's characterization?",
#     ...
# ]
#
# def run_eval_suite(questions: list[str]) -> pd.DataFrame:
#     ...